## Divisão da planilha 'dataset_pre_processado.csv' em 7 novas planilhas

**Entrada:** `data/dataset_pre_processado.csv`, gerado por `expl_preprocessamenro_hipote.ipynb`.

**Saída:** uma planilha por `key_area` em `data/<key_area>_data.csv`, lidas pelos notebooks de modelo.

Os arquivos de `data/` não estão no repositório (dados confidenciais do parceiro);
veja a seção "Os notebooks" do README para obtê-los e para a ordem de execução.


In [ ]:
import pandas as pd

from pathlib import Path

# ---- Configuração ----
INPUT_FILE = Path('../data/dataset_pre_processado.csv')
# Os sete CSVs gerados ficam diretamente em data/, na raiz do projeto.
OUTPUT_DIR = Path('../data')
OUTPUT_DIR.mkdir(exist_ok=True)

# Colunas removidas de todos os arquivos de saída, além de 'key_area'
# (decisão do grupo — ver Card H)
COLUMNS_TO_DROP = ['run_id', 'tc_number', 'reason', 'active_limit', 'key_area']

# Componente excluído do Mainback: identificado como erro de sensor
# (ver discussão de outliers / seção 4.2.2 da documentação)
MAINBACK_EXCLUDE_TC = 'Oven RTD Connector Body'

In [ ]:
def split_by_key_area(df: pd.DataFrame, output_dir: Path) -> tuple[dict, dict]:

    summary = {}
    area_dataframes = {}
    key_areas = df['key_area'].unique()
    print(f'Áreas encontradas ({len(key_areas)}): {list(key_areas)}\n')

    for area in key_areas:
        filename = output_dir / f"{area.lower().replace(' ', '_')}_data.csv"
        area_df = df[df['key_area'] == area].copy()

        if area == 'Mainback' and 'tc_name' in area_df.columns:
            area_df = area_df[area_df['tc_name'] != MAINBACK_EXCLUDE_TC]

        area_df = area_df.drop(columns=[c for c in COLUMNS_TO_DROP if c in area_df.columns])

        if 'temp' in area_df.columns:
            ordered_cols = [c for c in area_df.columns if c != 'temp'] + ['temp']
            area_df = area_df[ordered_cols]

        area_df.to_csv(filename, index=False)
        area_dataframes[area] = area_df
        summary[area] = len(area_df)
        print(f'"{filename.name}" salvo com {len(area_df)} linhas.')

    return summary, area_dataframes

In [ ]:
df = pd.read_csv(INPUT_FILE)
df['deflector_hole'] = df['deflector_hole'].replace('Open - Updated', 'Open')
print(f"Dataset principal carregado: {len(df)} linhas, {df['key_area'].nunique()} key_areas.\n")

summary, key_area_dataframes = split_by_key_area(df, OUTPUT_DIR)

# DataFrames finais, um para cada key_area.
mainback_df = key_area_dataframes['Mainback']
coil_temps_df = key_area_dataframes['Coil Temps']
main_vent_sys_df = key_area_dataframes['Main Vent Sys']
ipc_cooling_df = key_area_dataframes['IPC Cooling']
ipc_comp_df = key_area_dataframes['IPC Comp']
oven_hmi_df = key_area_dataframes['Oven HMI']
ct_knobs_df = key_area_dataframes['CT Knobs']

In [ ]:
# ---- Verificação de integridade ----
total_saved = sum(summary.values())
excluded_rows = len(df[df['tc_name'] == MAINBACK_EXCLUDE_TC]) if 'tc_name' in df.columns else 0
expected = len(df) - excluded_rows

print('Resumo por área:')
for area, n in summary.items():
    print(f'  {area}: {n} linhas')

print(f"\nTotal salvo: {total_saved} | Esperado (total original - excluídas): {expected}")
assert total_saved == expected, 'Divergência na contagem de linhas! Verifique o processamento.'
print('Contagem de linhas conferida com sucesso — nenhuma linha perdida ou duplicada além do esperado.')

In [ ]:
df.head(100)

## Converter 'tc_name' em colunas

In [ ]:
def tc_names_to_columns(area_df: pd.DataFrame) -> pd.DataFrame:
    """Transforma tc_name em colunas, preservando todas as temperaturas."""
    index_columns = [column for column in area_df.columns if column not in ['tc_name', 'temp']]
    pivot_source = area_df.copy()

    # Numera medições repetidas para não descartar temperaturas no pivot.
    pivot_source['measurement_number'] = (
        pivot_source.groupby(index_columns + ['tc_name'], dropna=False).cumcount()
    )

    wide_df = (
        pivot_source.pivot(
            index=index_columns + ['measurement_number'],
            columns='tc_name',
            values='temp',
        )
        .reset_index()
    )
    wide_df.columns.name = None

    # A numeração é apenas auxiliar para o pivot e não faz parte da planilha final.
    wide_df = wide_df.drop(columns='measurement_number')


    return wide_df

key_area_wide_dataframes = {
    area: tc_names_to_columns(area_df)
    for area, area_df in key_area_dataframes.items()
}

mainback_wide_df = key_area_wide_dataframes['Mainback']
coil_temps_wide_df = key_area_wide_dataframes['Coil Temps']
main_vent_sys_wide_df = key_area_wide_dataframes['Main Vent Sys']
ipc_cooling_wide_df = key_area_wide_dataframes['IPC Cooling']
ipc_comp_wide_df = key_area_wide_dataframes['IPC Comp']
oven_hmi_wide_df = key_area_wide_dataframes['Oven HMI']
ct_knobs_wide_df = key_area_wide_dataframes['CT Knobs']

for area, wide_df in key_area_wide_dataframes.items():
    filename = OUTPUT_DIR / f"{area.lower().replace(' ', '_')}_data.csv"
    wide_df.to_csv(filename, index=False)
    print(f'"{filename.name}" salvo com {len(wide_df)} linhas e {len(wide_df.columns)} colunas.')

## Tratamento de dados nulos

Será utilizado a mediana caso a coluna do componente contenha dados nulos, caso contrário será utilizado a média.

In [ ]:
def fill_component_nulls(wide_df: pd.DataFrame, component_columns) -> dict:
    """Fill temperature nulls using median for IQR outliers, otherwise mean."""
    # Guarda qual método foi usado em cada coluna de componente.
    treatment_by_component = {}

    # Analisa cada componente (cada tc_name virou uma coluna).
    for component in component_columns:
        # Converte valores para numéricos; valores inválidos passam a ser nulos.
        temperatures = pd.to_numeric(wide_df[component], errors='coerce')
        valid_temperatures = temperatures.dropna()
        nulls_changed = int(temperatures.isna().sum())

        # Sem valores válidos, não é possível calcular média ou mediana.
        if valid_temperatures.empty:
            treatment_by_component[component] = {
                'method': 'not applied - no valid values',
                'replacement_temperature': None,
                'nulls_changed': 0,
            }
            continue

        # Calcula os quartis e o IQR para identificar outliers.
        q1 = valid_temperatures.quantile(0.25)
        q3 = valid_temperatures.quantile(0.75)
        iqr = q3 - q1
        lower_limit = q1 - 1.5 * iqr
        upper_limit = q3 + 1.5 * iqr
        has_outliers = ((valid_temperatures < lower_limit) | (valid_temperatures > upper_limit)).any()

        # Com outliers, a mediana é mais robusta; sem eles, usa-se a média.
        if has_outliers:
            fill_value = valid_temperatures.median()
            method = 'median (with outliers)'
        else:
            fill_value = valid_temperatures.mean()
            method = 'mean (without outliers)'

        # Preenche somente as temperaturas nulas da coluna atual.
        wide_df[component] = temperatures.fillna(fill_value)
        treatment_by_component[component] = {
            'method': method,
            'replacement_temperature': fill_value,
            'nulls_changed': nulls_changed,
        }

    return treatment_by_component

# Aplica o tratamento às sete planilhas, usando apenas as colunas de componentes.
null_treatment_summary = {}
for area, wide_df in key_area_wide_dataframes.items():
    component_columns = key_area_dataframes[area]['tc_name'].dropna().unique()
    null_treatment_summary[area] = fill_component_nulls(wide_df, component_columns)

    # Salva a planilha da área já com as temperaturas nulas tratadas.
    filename = OUTPUT_DIR / f"{area.lower().replace(' ', '_')}_data.csv"
    wide_df.to_csv(filename, index=False)
    print(f'"{filename.name}" updated after null-temperature treatment.')

In [ ]:
# Reúne apenas os componentes que tiveram valores nulos substituídos.
replacement_records = []
for area, components in null_treatment_summary.items():
    for component, details in components.items():
        if details['nulls_changed'] > 0:
            replacement_records.append({
                'Seção do fogão': area,
                'Componente': component,
                'Temperatura utilizada': details['replacement_temperature'],
                'Método': details['method'],
                'Casas nulas alteradas': details['nulls_changed'],
            })

replacement_report_df = pd.DataFrame(
    replacement_records,
    columns=[
        'Seção do fogão',
        'Componente',
        'Temperatura utilizada',
        'Método',
        'Casas nulas alteradas',
    ],
)

if replacement_report_df.empty:
    print('Nenhum valor nulo foi substituído.')
else:
    display(replacement_report_df)

## Converter categorias 'Não Informado' pela moda

In [ ]:
# Substitui "Não informado" por "No losses" somente quando losses é a única coluna com essa categoria.
TEST_TYPE_COLUMNS = [
    'blower_direction',
    'housing',
    'board_position',
    'run_case',
    'losses',
    'deflector_hole',
    'material_coil_support',
]
MISSING_CATEGORY = 'Não informado'

def is_missing_category(series: pd.Series) -> pd.Series:
    """Identifica 'Não informado' mesmo com espaços extras."""
    return series.astype('string').str.strip().str.casefold().eq(MISSING_CATEGORY.casefold())

columns_with_missing_category = sorted({
    column
    for wide_df in key_area_wide_dataframes.values()
    for column in TEST_TYPE_COLUMNS
    if column in wide_df.columns and is_missing_category(wide_df[column]).any()
})

category_replacement_records = []

if columns_with_missing_category == ['losses']:
    replacement_value = 'No losses'

    for area, wide_df in key_area_wide_dataframes.items():
        if 'losses' not in wide_df.columns:
            continue

        missing_mask = is_missing_category(wide_df['losses'])
        if not missing_mask.any():
            continue

        wide_df.loc[missing_mask, 'losses'] = replacement_value

        # run_id não está nas planilhas finais; recupera os runs afetados no dataset original.
        affected_runs = (
            df.loc[
                df['key_area'].eq(area) & is_missing_category(df['losses']),
                'run_id',
            ]
            .dropna()
            .drop_duplicates()
            .tolist()
        )

        for run_id in affected_runs:
            category_replacement_records.append({
                'Planilha': area,
                'Run alterado': run_id,
                'Tipo de teste': 'losses',
                'Valor inserido': replacement_value,
            })

        # Persiste apenas as planilhas que receberam a correção.
        filename = OUTPUT_DIR / f"{area.lower().replace(' ', '_')}_data.csv"
        wide_df.to_csv(filename, index=False)
else:
    print(
        'Nenhuma alteração realizada: as colunas com Não informado são ' 
        f"{columns_with_missing_category}; esperada apenas ['losses']."
    )

category_replacement_report_df = pd.DataFrame(
    category_replacement_records,
    columns=['Planilha', 'Run alterado', 'Tipo de teste', 'Valor inserido'],
)

if category_replacement_report_df.empty:
    print("Nenhum valor 'Não informado' foi encontrado nos tipos de teste.")
else:
    display(category_replacement_report_df.sort_values(
        ['Planilha', 'Run alterado', 'Tipo de teste']
    ).reset_index(drop=True))


## Converter IGBTs 1 a 16 por apenas 1, utilizando a média

In [ ]:
import re

# Consolida IGBT 1 a IGBT 16 em uma única coluna. A média é calculada
# por linha (isto é, para cada configuração/run) e ignora valores ausentes.
PADRAO_IGBT = re.compile(r'^IGBT (?:[1-9]|1[0-6])$')

for area, wide_df in key_area_wide_dataframes.items():
    colunas_igbt = [coluna for coluna in wide_df.columns if PADRAO_IGBT.fullmatch(coluna)]

    if not colunas_igbt:
        continue

    # Converte para número para garantir que a média seja calculada corretamente.
    media_igbt = wide_df[colunas_igbt].apply(pd.to_numeric, errors='coerce').mean(axis=1).round(2)

    # Mantém a nova coluna na posição da primeira coluna IGBT original.
    posicao_igbt = min(wide_df.columns.get_loc(coluna) for coluna in colunas_igbt)
    wide_df = wide_df.drop(columns=colunas_igbt)
    wide_df.insert(posicao_igbt, 'IGBT', media_igbt)

    key_area_wide_dataframes[area] = wide_df
    filename = OUTPUT_DIR / f"{area.lower().replace(' ', '_')}_data.csv"
    wide_df.to_csv(filename, index=False)
    if area == 'IPC Comp':
        print(f'IPC Comp: {len(colunas_igbt)} colunas consolidadas em IGBT e salvo em {filename.name}.')


## Unificar dados com configurações iguais

In [ ]:
# Unifica medições que possuem a mesma configuração de teste.
# Para cada componente, a temperatura final é a média das medições repetidas.
CONFIGURATION_COLUMNS = [
    'blower_direction',
    'housing',
    'board_position',
    'run_case',
    'losses',
    'deflector_hole',
    'material_coil_support',
]

unification_summary = {}

for area, wide_df in key_area_wide_dataframes.items():
    missing_columns = set(CONFIGURATION_COLUMNS) - set(wide_df.columns)
    if missing_columns:
        raise KeyError(f'{area}: colunas de configuração ausentes: {sorted(missing_columns)}')

    temperature_columns = [
        column for column in wide_df.columns
        if column not in CONFIGURATION_COLUMNS
    ]

    # Garante o cálculo da média mesmo se alguma temperatura estiver como texto.
    temperatures = wide_df[temperature_columns].apply(pd.to_numeric, errors='coerce')
    grouped_df = (
        pd.concat([wide_df[CONFIGURATION_COLUMNS], temperatures], axis=1)
        .groupby(CONFIGURATION_COLUMNS, dropna=False, as_index=False)[temperature_columns]
        .mean()
    )

    unification_summary[area] = {
        'linhas_antes': len(wide_df),
        'linhas_depois': len(grouped_df),
        'configurações_unificadas': len(wide_df) - len(grouped_df),
    }
    key_area_wide_dataframes[area] = grouped_df

    filename = OUTPUT_DIR / f"{area.lower().replace(' ', '_')}_data.csv"
    grouped_df.to_csv(filename, index=False)
    print(
        f'{area}: {len(wide_df)} → {len(grouped_df)} linhas '
        f'({len(wide_df) - len(grouped_df)} configurações repetidas unificadas).'
    )

# Atualiza os DataFrames individuais para refletir os dados unificados.
mainback_wide_df = key_area_wide_dataframes['Mainback']
coil_temps_wide_df = key_area_wide_dataframes['Coil Temps']
main_vent_sys_wide_df = key_area_wide_dataframes['Main Vent Sys']
ipc_cooling_wide_df = key_area_wide_dataframes['IPC Cooling']
ipc_comp_wide_df = key_area_wide_dataframes['IPC Comp']
oven_hmi_wide_df = key_area_wide_dataframes['Oven HMI']
ct_knobs_wide_df = key_area_wide_dataframes['CT Knobs']

display(pd.DataFrame(unification_summary).T)
